# 03 — Feature Engineering

---

This notebook creates the first set of **pre-match features** for football match prediction.

## Objectives

The main goal is to transform the clean base match dataset into a model-ready dataset containing features that describe team form and recent performance **before each match**.

In particular, this notebook will:

- convert match-level data into team-level longitudinal format,
- create match outcome variables such as points won,
- compute rolling statistics based only on past matches,
- merge home-team and away-team features back into one match-level table,
- create difference features between home and away teams,
- save the resulting pre-match feature dataset for modeling.

## Important methodological principle

A critical rule in predictive modeling is to avoid **data leakage**.

For this reason, all rolling features in this notebook are computed using only matches played **before** the current match.  
This is achieved by shifting each time series by one match before applying rolling averages.

This means that for every match, the model will only see information that would have been available before kick-off.

---

## 1. Imports and setup

We begin by importing the required libraries and project functions.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from src.utils import ensure_directories
from src.feature_engineering import (
    add_match_outcome_points,
    long_format_team_matches,
    add_basic_team_features,
    add_rolling_features,
    split_home_away_features,
    merge_match_features,
    add_difference_features,
)

---

## 2. Load the base match dataset

We now load the clean match-level dataset prepared in the previous notebook.

This table already contains:
- one row per match,
- final score,
- xG values,
- target variables,
- chronological ordering.

In [2]:
base_path = INTERIM_DATA_DIR / "base_matches.parquet"

if not base_path.exists():
    base_path = INTERIM_DATA_DIR / "base_matches.csv"

df_matches = pd.read_parquet(base_path) if base_path.suffix == ".parquet" else pd.read_csv(base_path)
df_matches["date"] = pd.to_datetime(df_matches["date"], errors="coerce")

---

## 3. Inspect the input dataset

Before constructing features, we briefly inspect the data structure.
This is useful for verifying that the base table contains the expected columns and data types.

In [3]:
print("Shape:", df_matches.shape)
display(df_matches.head())

Shape: (882, 36)


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,home_team_code,away_team_code,...,away_ppda,home_deep_completions,away_deep_completions,target_1x2,home_win,draw,away_win,round,week,matchday
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,19.714286,2,16,A,0,0,1,Bundesliga,1,1
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,12.105263,5,4,H,1,0,0,Bundesliga,1,1
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,16.578947,8,7,H,1,0,0,Bundesliga,1,1
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,21.727273,8,7,A,0,0,1,Bundesliga,1,1
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,19.5,12,7,D,0,1,0,Bundesliga,1,1


---

## 4. Add match outcome points

A convenient way to summarize match results for later rolling form features is to convert outcomes into points:

- 3 points for a win,
- 1 point for a draw,
- 0 points for a loss.

We add points separately for the home team and the away team.

In [4]:
df_matches = add_match_outcome_points(df_matches)
display(df_matches[[
    "date", "home_team", "away_team",
    "home_goals", "away_goals",
    "home_points", "away_points"
]].head())

,date,home_team,away_team,home_goals,away_goals,home_points,away_points
0,2023-08-18 18:30:00,Werder Bremen,Bayern Munich,0,4,0.0,3.0
1,2023-08-19 13:30:00,Bayer Leverkusen,RasenBallsport Leipzig,3,2,3.0,0.0
2,2023-08-19 13:30:00,Wolfsburg,FC Heidenheim,2,0,3.0,0.0
3,2023-08-19 13:30:00,Hoffenheim,Freiburg,1,2,0.0,3.0
4,2023-08-19 13:30:00,Augsburg,Borussia M.Gladbach,4,4,1.0,1.0


---

## 5. Convert match-level data to team-level long format

At the moment, each row corresponds to one match.

However, rolling form features are easier to compute when each row corresponds to:

- one team,
- in one match,
- with its own goals, xG, points, and opponent.

Therefore, we reshape the data into a **long format**:
- one row for the home team,
- one row for the away team,
- for every match.

In [5]:
df_long = long_format_team_matches(df_matches)
display(df_long.head(10))

,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,np_xg_for,np_xg_against,expected_points,ppda,deep_completions,points,is_home
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,1.77704,1.16071,1.8311,11.176471,12,1.0,1
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,0.726241,2.06518,0.201,16.545455,6,0.0,0
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,1.88106,1.86494,1.3996,18.0,3,1.0,1
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,1.34046,1.46279,1.2679,17.9,7,0.0,0
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,1.04683,0.487325,1.9351,13.363636,2,3.0,1
5,23118,2023-10-01 15:30:00,2023,3,Augsburg,Freiburg,0,2,1.00081,1.2721,1.00081,0.51432,1.0663,11.565217,2,0.0,0
6,23122,2023-10-07 13:30:00,2023,3,Augsburg,Darmstadt,1,2,1.18473,1.96153,1.18473,1.20376,0.8073,9.64,12,0.0,1
7,23136,2023-10-22 15:30:00,2023,3,Augsburg,FC Heidenheim,5,2,2.82353,1.99327,2.06575,1.99327,1.9913,22.8125,4,3.0,0
8,23144,2023-10-28 13:30:00,2023,3,Augsburg,Wolfsburg,3,2,1.59374,1.25516,1.59374,0.497379,1.6435,16.736842,5,3.0,1
9,23151,2023-11-04 14:30:00,2023,3,Augsburg,FC Cologne,1,1,2.54685,2.22618,2.54685,2.22618,1.6295,16.0,4,1.0,0


--- 

## 6. Add basic team-level performance features

We now derive simple team-level variables that summarize each team's match performance:

- goal difference,
- xG difference.

These variables are useful because they often capture performance more effectively than raw goals or xG alone.

In [6]:
df_long = add_basic_team_features(df_long)
display(df_long.head())

,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,np_xg_for,np_xg_against,expected_points,ppda,deep_completions,points,is_home,goal_diff,xg_diff,np_xg_diff
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,1.77704,1.16071,1.8311,11.176471,12,1.0,1,0,0.61633,0.61633
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,0.726241,2.06518,0.201,16.545455,6,0.0,0,-2,-2.096719,-1.338939
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,1.88106,1.86494,1.3996,18.0,3,1.0,1,0,0.01612,0.01612
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,1.34046,1.46279,1.2679,17.9,7,0.0,0,-3,-0.12233,-0.12233
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,1.04683,0.487325,1.9351,13.363636,2,3.0,1,1,0.559505,0.559505


---

## 7. Create rolling pre-match features

This is the key part of the notebook.

For each team, we compute rolling averages over past matches for variables such as:

- goals scored,
- goals conceded,
- xG for,
- xG against,
- points,
- goal difference,
- xG difference.

### Why rolling features?

In football prediction, recent team form is often more informative than full-season averages.
Rolling features summarize this recent form in a compact and model-friendly way.

### Why use `shift(1)`?

Without shifting, the current match would be included in its own feature calculation, which would cause leakage.

By using `shift(1)`, we ensure that the rolling window contains only matches played **before** the current one.

In [7]:
df_long = add_rolling_features(df_long, windows=[3, 5])
display(df_long.head(10))

,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,...,points_ewm_span_5,goal_diff_avg_last_3,goal_diff_avg_last_5,goal_diff_ewm_span_5,xg_diff_avg_last_3,xg_diff_avg_last_5,xg_diff_ewm_span_5,np_xg_diff_avg_last_3,np_xg_diff_avg_last_5,np_xg_diff_ewm_span_5
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,...,1.000000,0.000000,0.000000,0.000000,0.616330,0.616330,0.616330,0.616330,0.616330,0.616330
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,...,0.666667,-1.000000,-1.000000,-0.666667,-0.740195,-0.740195,-0.288020,-0.361304,-0.361304,-0.035426
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,...,0.777778,-0.666667,-0.666667,-0.444444,-0.488090,-0.488090,-0.186640,-0.235496,-0.235496,-0.018244
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,...,0.518519,-1.666667,-1.250000,-1.296296,-0.734310,-0.396650,-0.165203,-0.481716,-0.207205,-0.052939
5,23118,2023-10-01 15:30:00,2023,3,Augsburg,Freiburg,0,2,1.00081,1.2721,...,1.345679,-0.666667,-0.800000,-0.530864,0.151098,-0.205419,0.076366,0.151098,-0.053863,0.151209
6,23122,2023-10-07 13:30:00,2023,3,Augsburg,Darmstadt,1,2,1.18473,1.96153,...,0.897119,-1.333333,-1.200000,-1.020576,0.055295,-0.382943,-0.039519,0.307888,-0.079831,0.262969
7,23136,2023-10-22 15:30:00,2023,3,Augsburg,FC Heidenheim,5,2,2.82353,1.99327,...,0.598080,-0.666667,-1.000000,-1.013717,-0.162862,-0.118959,-0.285279,0.342322,0.184151,0.168969
8,23144,2023-10-28 13:30:00,2023,3,Augsburg,Wolfsburg,3,2,1.59374,1.25516,...,1.398720,0.000000,-0.400000,0.324188,-0.072610,0.043869,0.086567,0.179980,0.195423,0.136806
9,23151,2023-11-04 14:30:00,2023,3,Augsburg,FC Cologne,1,1,2.54685,2.22618,...,1.932480,1.000000,0.400000,0.549459,0.130680,0.136051,0.170571,0.383270,0.439161,0.456658


---

## 8. Inspect rolling features for one team

To verify that rolling features behave as expected, it is useful to inspect one specific team over time.

This helps confirm that:
- the first match has no prior history,
- later matches use only past information,
- the feature values evolve logically over the season.

In [8]:
team_example = df_long["team"].iloc[0]
df_long[df_long["team"] == team_example][[
    "date",
    "team",
    "opponent",
    "goals_for",
    "goals_against",
    "points",
    "goals_for_avg_last_3",
    "xg_for_avg_last_3",
    "points_avg_last_3",
]].head(10)

,date,team,opponent,goals_for,goals_against,points,goals_for_avg_last_3,xg_for_avg_last_3,points_avg_last_3
0,2023-08-19 13:30:00,Augsburg,Borussia M.Gladbach,4,4,1.0,NaN,NaN,NaN
1,2023-08-27 15:30:00,Augsburg,Bayern Munich,1,3,0.0,4.000000,2.534820,1.000000
2,2023-09-02 13:30:00,Augsburg,Bochum,2,2,1.0,2.500000,1.630530,0.500000
3,2023-09-16 13:30:00,Augsburg,RasenBallsport Leipzig,0,3,0.0,2.333333,1.714040,0.666667
4,2023-09-23 13:30:00,Augsburg,Mainz 05,2,1,3.0,1.000000,1.315920,0.333333
5,2023-10-01 15:30:00,Augsburg,Freiburg,0,2,0.0,1.333333,1.422783,1.333333
6,2023-10-07 13:30:00,Augsburg,Darmstadt,1,2,0.0,0.666667,1.129367,1.000000
7,2023-10-22 15:30:00,Augsburg,FC Heidenheim,5,2,3.0,1.000000,1.077457,1.000000
8,2023-10-28 13:30:00,Augsburg,Wolfsburg,3,2,3.0,2.000000,1.669690,1.000000
9,2023-11-04 14:30:00,Augsburg,FC Cologne,1,1,1.0,3.000000,1.867333,2.000000


---

## 9. Split home and away team feature tables

The rolling features were created in team-level long format.

To build the final modeling dataset, we now split the table into:

- one table containing home-team features,
- one table containing away-team features.

Both tables will later be merged back into the original match-level dataset.

In [9]:
home_features, away_features = split_home_away_features(df_long)

print("Home feature table:", home_features.shape)
print("Away feature table:", away_features.shape)

Home feature table: (882, 59)
Away feature table: (882, 59)


---

## 10. Merge pre-match features back into the match-level dataset

We now combine the home-side and away-side rolling features into a single match-level dataset.

At this point, each row still represents one match, but now includes:
- home-team recent form,
- away-team recent form,
- season context for both sides.


In [10]:
df_features = merge_match_features(df_matches, home_features, away_features)

print("Feature dataset shape:", df_features.shape)
display(df_features.head())

Feature dataset shape: (882, 152)


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team_x,away_team_x,home_team_code,away_team_code,...,away_points_ewm_span_5,away_goal_diff_avg_last_3,away_goal_diff_avg_last_5,away_goal_diff_ewm_span_5,away_xg_diff_avg_last_3,away_xg_diff_avg_last_5,away_xg_diff_ewm_span_5,away_np_xg_diff_avg_last_3,away_np_xg_diff_avg_last_5,away_np_xg_diff_ewm_span_5
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


--- 

## 11. Create difference features

In football match prediction, the relative strength of the two teams is often more informative than their absolute values.

For this reason, we create difference features of the form:

`home_feature - away_feature`

These variables directly express home-side advantage or disadvantage in recent form and performance.

In [11]:
difference_base_names = [
    "goals_for_avg_last_3",
    "goals_against_avg_last_3",
    "xg_for_avg_last_3",
    "xg_against_avg_last_3",
    "points_avg_last_3",
    "goal_diff_avg_last_3",
    "xg_diff_avg_last_3",
    "goals_for_avg_last_5",
    "goals_against_avg_last_5",
    "xg_for_avg_last_5",
    "xg_against_avg_last_5",
    "points_avg_last_5",
    "goal_diff_avg_last_5",
    "xg_diff_avg_last_5",
]

df_features = add_difference_features(df_features, difference_base_names)
display(df_features.head())

,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team_x,away_team_x,home_team_code,away_team_code,...,diff_points_avg_last_3,diff_goal_diff_avg_last_3,diff_xg_diff_avg_last_3,diff_goals_for_avg_last_5,diff_goals_against_avg_last_5,diff_xg_for_avg_last_5,diff_xg_against_avg_last_5,diff_points_avg_last_5,diff_goal_diff_avg_last_5,diff_xg_diff_avg_last_5
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

## 12. Missing values in early matches

Rolling features naturally produce missing values at the very beginning of a team's season because there are not enough prior matches.

This is expected and not an error.

At the modeling stage, we will decide how to handle these early-season missing values. Possible strategies include:
- keeping them and using models that can handle missingness,
- imputing them,
- or dropping the earliest matches when necessary.

In [12]:
missing_share = df_features.isna().mean().sort_values(ascending=False)
missing_share.head(20).to_frame("missing_share")

,missing_share
round,0.420635
matchday,0.148526
week,0.148526
diff_goal_diff_avg_last_3,0.013605
diff_goals_for_avg_last_3,0.013605
diff_goals_against_avg_last_3,0.013605
diff_xg_for_avg_last_3,0.013605
diff_xg_against_avg_last_3,0.013605
diff_points_avg_last_3,0.013605
diff_xg_diff_avg_last_5,0.013605


---

## 13. Select the initial modeling feature set

At this stage, we define the first feature subset that will be used in later modeling notebooks.

This is only a starting point.
More advanced features such as ELO, home/away-specific rolling splits, and player-related features can be added later.

In [13]:
initial_feature_columns = [
    "game_id",
    "date",
    "season",
    "league",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "target_1x2",
    "home_win",
    "draw",
    "away_win",
]

rolling_feature_columns = [col for col in df_features.columns if "avg_last_" in col]
difference_feature_columns = [col for col in df_features.columns if col.startswith("diff_")]

final_columns = initial_feature_columns + rolling_feature_columns + difference_feature_columns
final_columns = [col for col in final_columns if col in df_features.columns]

df_model_input = df_features[final_columns].copy()

print("Model input shape:", df_model_input.shape)
display(df_model_input.head())

Model input shape: (882, 90)


,game_id,date,home_goals,away_goals,home_xg,away_xg,target_1x2,home_win,draw,away_win,...,diff_points_avg_last_3,diff_goal_diff_avg_last_3,diff_xg_diff_avg_last_3,diff_goals_for_avg_last_5,diff_goals_against_avg_last_5,diff_xg_for_avg_last_5,diff_xg_against_avg_last_5,diff_points_avg_last_5,diff_goal_diff_avg_last_5,diff_xg_diff_avg_last_5
0,23065,2023-08-18 18:30:00,0,4,0.63974,2.89704,A,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23066,2023-08-19 13:30:00,3,2,1.73279,1.60393,H,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23067,2023-08-19 13:30:00,2,0,3.21449,1.17776,H,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,23068,2023-08-19 13:30:00,1,2,1.67186,3.24821,A,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,23069,2023-08-19 13:30:00,4,4,2.53482,1.91849,D,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


--- 

## 14. Save the feature dataset

We save the resulting dataset for use in the modeling notebooks.

In [14]:
ensure_directories([PROCESSED_DATA_DIR])

output_path = PROCESSED_DATA_DIR / "match_features.parquet"

try:
    df_model_input.to_parquet(output_path, index=False)
except Exception:
    output_path = PROCESSED_DATA_DIR / "match_features.csv"
    df_model_input.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: C:\Users\cerve\Desktop\DP\match_prediction\data\processed\match_features.csv


---

## 15. Summary

This notebook created the first version of the pre-match feature dataset.

### What was done

- match results were converted into points,
- the data were reshaped into team-level long format,
- rolling performance features were computed using only past matches,
- home-team and away-team features were merged back into one match-level table,
- difference features were created,
- the resulting dataset was saved for later modeling.

### Why this step is important

This notebook transforms raw match data into a representation that can actually be used in predictive models.

Instead of predicting outcomes from isolated match records, we now describe each match through the **recent performance history** of both teams.

### Next step

The next notebook will extend the feature set further, most likely by adding:

- ELO ratings,
- home/away specific rolling form,
- and possibly additional contextual variables.